# Qwen Image 2.1 — Colabで透過画像を生成（T4省メモリ版）
**旧ノートブックでCUDA OOMが出た場合は、まずランタイムを再起動してください。** 古いpipeをGPUに残したままモデル読み込みセルを再実行しないでください。GitHubの最新版から開き直し、上から実行します。

T4向けに、層単位のCPUオフロード・VAEタイル分割・512px・KVキャッシュ無効化を採用します。転送が増えるため低速になります。モデルはCPUにも保持するため、通常RAMにも余裕が必要です。T4実機での修正版実行は未検証で、無料Colabでの動作保証ではありません。

[公式モデル](https://github.com/QwenLM/Qwen-Image-2.1) / [Diffusers省メモリ手順](https://huggingface.co/docs/diffusers/optimization/memory)


In [ ]:
%pip install "torch>=2.4.0" "transformers>=5.17" accelerate pillow psutil
%pip install git+https://github.com/huggingface/diffusers


インストール後に再起動を求められたら再起動し、次のセルから進みます。GPUはランタイム設定で選択してください。

In [ ]:
import gc, torch, psutil, shutil
assert torch.cuda.is_available(), 'GPUランタイムを選択してください。'
print('GPU:', torch.cuda.get_device_name(0))
free, total = torch.cuda.mem_get_info()
print('VRAM 空き/合計 GiB:', round(free / 2**30, 1), '/', round(total / 2**30, 1))
ram = psutil.virtual_memory()
print('通常RAM 空き/合計 GiB:', round(ram.available / 2**30, 1), '/', round(ram.total / 2**30, 1))
print('ディスク空き GiB:', round(shutil.disk_usage('/content').free / 2**30, 1))
if ram.available < 36 * 2**30:
    print('注意：非量子化のモデルは通常RAMを数十GB使います。高RAM環境を検討してください。36GiBは注意表示の目安で、必要量の保証ではありません。')


## モデルを1つだけ読み込む
`sequential`では層ごとにGPUへ転送します。`.to("cuda")`を追加しないでください。セルをやり直す前にランタイムを再起動してください。

In [ ]:
import gc
import os
import torch
from diffusers import QwenImage21Pipeline


def load_pipeline(memory_mode=None):
    if not torch.cuda.is_available():
        raise RuntimeError('CUDA GPUが必要です。GPUランタイムを選択してください。')
    # Model offload moves whole components: the ~8B text encoder or DiT can
    # exhaust a T4 even though other components are held on the CPU.
    mode = memory_mode or os.environ.get('QWEN_MEMORY_MODE', 'cuda' if os.environ.get('QWEN_CPU_OFFLOAD') == '0' else 'model')
    if mode == 'auto':
        mode = 'sequential' if torch.cuda.get_device_properties(0).total_memory < 24 * 2**30 else 'model'
    if mode not in ('sequential', 'model', 'cuda'):
        raise ValueError('memory_modeはsequential / model / cuda / autoです。')
    gc.collect()
    torch.cuda.empty_cache()
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    pipe = QwenImage21Pipeline.from_pretrained(
        'Qwen/Qwen-Image-2.1', torch_dtype=dtype, low_cpu_mem_usage=True)
    # Qwen's RGBA VAE supports tiled decoding; slicing is not assumed supported.
    pipe.vae.enable_tiling()
    if mode == 'sequential':
        # Keep weights on CPU and move submodules only when they are needed.
        # Never call .to('cuda') before attaching these hooks.
        pipe.enable_sequential_cpu_offload()
    elif mode == 'model':
        pipe.enable_model_cpu_offload()
    else:
        pipe.to('cuda')
    pipe._atlas_low_memory = mode == 'sequential'
    print('Memory mode:', mode, '/ VAE tiling: on / dtype:', dtype)
    return pipe


def generate(pipe, prompt, size=512, seed=42, steps=40, images=None):
    if not isinstance(size, int) or size < 256 or size > 2048 or size % 32:
        raise ValueError('サイズは256〜2048、32の倍数を指定してください。')
    low_memory = getattr(pipe, '_atlas_low_memory', False)
    if low_memory and size > 768:
        raise ValueError('省メモリモードは768px以下にしてください。T4ではまず512pxで実行します。')
    if low_memory and images and len(images) > 1:
        raise ValueError('省メモリモードの参照画像は1枚にしてください。')
    kwargs = dict(prompt=prompt, width=size, height=size, output_resolution=size,
                  num_inference_steps=steps, true_cfg_scale=1.0,
                  use_kv_cache=not low_memory,
                  generator=torch.Generator('cpu').manual_seed(seed))
    if images:
        kwargs['image'] = images
    gc.collect()
    torch.cuda.empty_cache()
    try:
        with torch.inference_mode():
            return pipe(**kwargs).images[0]
    except torch.cuda.OutOfMemoryError:
        # No automatic retry: keep size/seed and model changes explicit to users.
        pipe.maybe_free_model_hooks()
        gc.collect()
        torch.cuda.empty_cache()
        raise RuntimeError(
            'T4のGPUメモリが不足しました。ランタイムを再起動し、sequentialでモデルを1つだけ読み込み、'
            'size=384（さらに不足する場合は256）、参照画像なしで再実行してください。'
            '通常RAMの不足や別モデルのGPU使用も確認してください。') from None

if 'pipe' in globals():
    raise RuntimeError('pipeが既に存在します。読み込み直す場合はランタイムを再起動してください。')
pipe = load_pipeline(memory_mode='sequential')


## 512pxで生成
この実行では大きなKVキャッシュを保持せず、VAEはタイル単位で処理します。40ステップは画質設定です。ステップ数を減らすだけではピークメモリは大きく減りません。

In [ ]:
prompt = "This is an RGBA image with transparency. A cute cartoon dragon sticker. The image has alpha channel and the background is transparent."
size = 512
seed = 42
steps = 40
# Previous failed/successful output must not be downloaded as this run's result.
from pathlib import Path
Path('/content/qwen-transparent.png').unlink(missing_ok=True)
image = generate(pipe, prompt, size, seed, steps)
image.save('/content/qwen-transparent.png')
display(image)
print('Mode:', image.mode, 'Size:', image.size)
print('Alpha range:', image.getchannel('A').getextrema() if 'A' in image.getbands() else 'No alpha channel')


In [ ]:
from google.colab import files
from pathlib import Path
assert Path('/content/qwen-transparent.png').exists(), 'まず画像生成を正常完了してください。'
files.download('/content/qwen-transparent.png')


## 任意：参照画像を1枚だけ編集
T4ではまず1枚・512pxで確認します。入力も512px相当へ縮小して処理するため、原寸の細部は保持されない場合があります。

In [ ]:
from PIL import Image
import io
uploaded = files.upload()
assert len(uploaded) == 1, 'T4の省メモリモードでは参照画像は1枚です。'
reference = Image.open(io.BytesIO(next(iter(uploaded.values()))))
reference.thumbnail((512, 512))
edit_prompt = "This is an RGBA image with transparency. Extract the main subject and preserve its details. The image has alpha channel and the background is transparent."
Path('/content/qwen-edited.png').unlink(missing_ok=True)
edited = generate(pipe, edit_prompt, size, seed, steps, [reference])
edited.save('/content/qwen-edited.png')
display(edited)
files.download('/content/qwen-edited.png')


## それでもOOMになる場合
ランタイムを再起動し、参照画像なし・size=384または256で実行してください。画質・細部は低下します。通常RAM不足はGPUの省メモリ化だけでは解決しないため、高RAM環境や大容量GPUへの変更が必要です。完了後は「接続を解除してランタイムを削除」で資源を解放します。